# Week 13 Demo: Imputation and Dictionary Lookups


In Week 4, you learned how to **detect** data quality problems - missing values, duplicates, inconsistent formatting.
This demo picks up where Week 4 left off: given that you've already found problems, what do you actually do about them?

We'll cover two new techniques:
1. **Median imputation** - filling missing numeric values (global median in this demo, then category-specific pattern)
2. **Dictionary lookup imputation** - filling missing categorical values using a reference table

We'll also cover how to detect and fix **orphaned IDs** - references to rows that don't exist in another table.

**Dataset:** A HR dataset with 15 employees and a departments reference table.

## Setup

In [ ]:
import pandas as pd
import numpy as np

# Reference table: departments
departments = pd.DataFrame({
    'dept_id':   ['D01', 'D02', 'D03', 'D04', 'D05'],
    'dept_name': ['Engineering', 'Marketing', 'Finance', 'Operations', 'HR']
})

# Employee table — already "detected" as having three types of problems:
#   - E004 and E014: missing salary
#   - E008: salary of -15000 (implausible)
#   - E005 (D99) and E012 (D07): orphaned dept_id
employees = pd.DataFrame({
    'emp_id':    ['E001','E002','E003','E004','E005',
                  'E006','E007','E008','E009','E010',
                  'E011','E012','E013','E014','E015'],
    'name':      ['Alice Novak','Ben Carter','Clara Diaz','David Kim','Elena Marsh',
                  'Frank Osei','Grace Liu','Henry Patel','Iris Wong','James Reed',
                  'Karen Bell','Leo Nguyen','Maya Stone','Nathan Cruz','Olivia Park'],
    'dept_id':   ['D01','D02','D01','D03','D99',
                  'D02','D04','D01','D05','D03',
                  'D02','D07','D04','D01','D05'],
    'salary':    [72000, 58000, 91000, np.nan, 64000,
                  48000, 83000, -15000, 77000, 69000,
                  52000, 61000, 88000, np.nan, 74000],
    'hire_year': [2021, 2017, 2012, 2022, 2019,
                  2023, 2015, 2020, 2018, 2013,
                  2016, 2021, 2010, 2023, 2019]
})

print('Departments:')
print(departments)
print(f'\nEmployees: {len(employees)} rows')
print(employees)

---
## Quick Audit Recap

In a real workflow you'd run a full audit first (as you did in Week 4).
For this demo, we already know what's wrong. Let's confirm it quickly and move straight to fixes.

In [ ]:
# Confirm the three known problem types
valid_depts = set(departments['dept_id'])

print('1. Missing salaries:')
print(employees[employees['salary'].isnull()][['emp_id', 'name', 'salary']])

print('\n2. Implausible salary (negative):')
print(employees[employees['salary'] < 0][['emp_id', 'name', 'salary']])

print('\n3. Orphaned dept_id values:')
print(employees[~employees['dept_id'].isin(valid_depts)][['emp_id', 'name', 'dept_id']])

---
## Fix 1: Median Imputation for Missing Numeric Values

**Strategy:** Replace the negative salary with NaN (treating it as invalid), then impute all missing salaries using the median of the valid rows.

**In this HR example:** We use a **global** salary median across all employees.
**Later in this notebook:** We show the **category-specific** median pattern you will use when categories matter.

**Why median, not mean?** The mean is sensitive to outliers. If we accidentally include the -$15,000 salary in our calculation, it pulls the mean down unrealistically. The median ignores extremes and gives a more stable estimate.

In [ ]:
# Always work on a copy - preserve the original
employees_clean = employees.copy()

# Step 1: Replace the implausible (negative) salary with NaN
# Business rule: salary cannot be negative in this dataset; negative means data error
# Converting invalid values to NaN lets us treat them like other missing values
employees_clean.loc[employees_clean['salary'] < 0, 'salary'] = np.nan

print('Rows with missing salary after replacing negative:')
print(employees_clean[employees_clean['salary'].isnull()][['emp_id', 'name', 'salary']])

In [ ]:
# Step 2: Calculate the median from VALID rows only
# median() ignores NaN by default, so the replaced negative value is excluded
salary_median = employees_clean['salary'].median()
print(f'Median salary (valid rows only): ${salary_median:,.0f}')

# Compare to what the mean would have been with the bad value still in:
# mean() also ignores NaN by default, so replacing -15000 with NaN removes its influence
salary_mean_bad = employees['salary'].mean()  # includes -15000
salary_mean_clean = employees_clean['salary'].mean()  # excludes it after NaN replacement
print(f'Mean with bad value: ${salary_mean_bad:,.0f}')
print(f'Mean without bad value: ${salary_mean_clean:,.0f}')
print('\nMedian is more stable - this is why we prefer it for imputation.')

In [ ]:
# Step 3: Fill all missing salaries with the median
employees_clean['salary'] = employees_clean['salary'].fillna(salary_median)

# Verify
print('Missing salaries after imputation:', employees_clean['salary'].isnull().sum())
print('\nAll salaries now:')
print(employees_clean[['emp_id', 'name', 'salary']].to_string())

### What if categories matter?

In our HR dataset, imputing from the global median is reasonable.
But in a dataset with diverse categories — groceries, gas, rent, for example — you'd want to impute from the **category-specific** median.

A missing grocery amount should be replaced with the median grocery amount,
not the median of all transactions. Here's the pattern:

In [ ]:
# Category-specific median imputation (pattern preview)
# This is what you'll use in your assignment
# Note on sign convention: in transactions, expenses are often negative
# That is different from salary, where negative values are invalid

# Example: compute median only from valid GROC rows
# groc_median = df[
#     (df['category_id'] == 'GROC') &
#     df['amount'].between(-150, -20)  # keep plausible expense range only
# ]['amount'].median()
#
# Fill missing GROC amounts:
# mask = (df['category_id'] == 'GROC') & df['amount'].isnull()
# df.loc[mask, 'amount'] = groc_median

print('Pattern shown above - you will apply this in the assignment.')
print('Key points:')
print('  1. Calculate median from VALID rows only (exclude outliers)')
print('  2. Apply imputation only to rows in that category')
print('  3. Repeat for each category that has missing values')

---
## Fix 2: Dictionary Lookup for Orphaned IDs

**The problem:** Some employees have dept_id values (D99, D07) that don't exist in the departments table.

**The tool:** A Python dictionary maps a key to a value — exactly like a reference table maps an ID to a name.
We can build a dictionary from the departments table and use it to validate and enrich the employees table.

In [ ]:
# Build a lookup dictionary from the departments table
# dict(zip(keys, values)) is the standard pattern
dept_lookup = dict(zip(departments['dept_id'], departments['dept_name']))

print('Department lookup dictionary:')
print(dept_lookup)

In [ ]:
# Bracket notation: crashes with KeyError if key doesn't exist
print(dept_lookup['D01'])   # Works fine

# .get() with a default: safe for unknown keys
print(dept_lookup.get('D99', 'NOT FOUND'))  # Returns 'NOT FOUND'
print(dept_lookup.get('D07', 'NOT FOUND'))  # Returns 'NOT FOUND'

print('\nUse .get() when you don\'t know in advance which keys might be invalid.')

In [ ]:
# Validate the entire dept_id column using isin()
# Recompute valid_depts here to keep this section self-contained if run independently
valid_depts = set(departments['dept_id'])
orphan_mask = ~employees_clean['dept_id'].isin(valid_depts)

print(f'Employees with invalid dept_id: {orphan_mask.sum()}')
print(employees_clean[orphan_mask][['emp_id', 'name', 'dept_id']])

In [ ]:
# Fix: replace orphaned IDs with 'UNKNOWN'
# This is honest — we don't know their correct department
# rather than guess, we flag it explicitly
employees_clean.loc[orphan_mask, 'dept_id'] = 'UNKNOWN'

print('dept_id values after fix:')
print(employees_clean['dept_id'].value_counts())

In [ ]:
# Use .map() to add department names to the table
# Employees with UNKNOWN dept_id will get NaN — that's expected
employees_clean['dept_name'] = employees_clean['dept_id'].map(dept_lookup)

print('Employees with department names:')
print(employees_clean[['emp_id', 'name', 'dept_id', 'dept_name', 'salary']].to_string())

### What if we know the correct category from context?

In some cases — like the semester dataset — we CAN infer the correct value from another field.
If merchant M0001 is always FreshMart, and FreshMart always maps to GROC,
then any transaction at M0001 with a missing category should be GROC.

That's lookup imputation — using a dictionary to infer a missing categorical value.

In [ ]:
# Lookup imputation pattern (preview for your assignment)

# Build merchant_id -> category_id mapping
merchant_to_cat = {
    'M0001': 'GROC', 'M0002': 'GROC',   # FreshMart locations
    'M0009': 'DINE', 'M0010': 'DINE',   # BrewHouse locations
    'M0011': 'SUBS',                      # StreamFlix
    # ... all merchants
}

# If a transaction has an empty category_id,
# look up the correct category from its merchant_id
# mask = (df['category_id'] == '') | df['category_id'].isnull()
# df.loc[mask, 'category_id'] = (
#     df.loc[mask, 'merchant_id'].map(merchant_to_cat)
# )

print('Lookup imputation pattern:')
print('  1. Build a dict: merchant_id -> category_id')
print('  2. Find rows with missing category')
print('  3. Use .map() to fill from the merchant lookup')
print('\nYou will implement this fully in the assignment notebook.')

---
## Verify

Always re-run your checks after applying fixes to confirm everything is resolved.

In [ ]:
print('=== VERIFICATION ===')

# Check 1: No missing salaries
missing = employees_clean['salary'].isnull().sum()
print(f'Missing salaries: {missing}  (expected 0)')

# Check 2: No negative salaries
negative = (employees_clean['salary'] < 0).sum()
print(f'Negative salaries: {negative}  (expected 0)')

# Check 3: No orphaned dept_ids (excluding UNKNOWN which we set intentionally)
known_valid = valid_depts | {'UNKNOWN'}
still_orphaned = (~employees_clean['dept_id'].isin(known_valid)).sum()
print(f'Orphaned dept_ids remaining: {still_orphaned}  (expected 0)')

# Check 4: dept_name will be NaN for UNKNOWN dept_id after map(); this is expected
unknown_dept_names = employees_clean.loc[employees_clean['dept_id'] == 'UNKNOWN', 'dept_name'].isnull().sum()
print(f'UNKNOWN rows with missing dept_name: {unknown_dept_names}  (expected >= 0 and usually > 0)')

In [ ]:
# Save the cleaned dataset
from pathlib import Path

output_path = Path('employees_clean.csv').resolve()
employees_clean.to_csv(output_path, index=False)
print(f'Saved: {employees_clean.shape[0]} rows, {employees_clean.shape[1]} columns')
print(f'Output file: {output_path}')
print('\nFinal cleaned dataset:')
print(employees_clean.to_string())

---
## Summary

This demo covered two techniques that were not in Week 4:

**Median imputation:**
- Replace implausible values with NaN first
- Calculate the median from valid rows only
- Use `.fillna()` to fill all missing values at once
- For datasets with categories, compute a category-specific median

**Dictionary lookup:**
- Build a dict from a reference table using `dict(zip(keys, values))`
- Use `.isin()` to find orphaned IDs
- Use `.get()` for safe lookups that won't crash on missing keys
- Use `.map()` to apply the lookup across an entire column
- When you CAN infer the correct value from context, use lookup imputation
- When you CAN'T, flag with 'UNKNOWN' rather than guess

**You'll apply all of these patterns in the assignment notebook.**